# Vertebral Fracture Grader: Colab Data Preparation

This notebook prepares T4-L4 vertebra crops for a research model.

It is not a medical device and must not be used for clinical decisions.

Raw CT archives remain in temporary Colab storage. Only anonymous,
derived crops and manifests are saved to Google Drive.


In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("System:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
import os
from pathlib import Path

REPOSITORY_URL = "https://github.com/kvskushal/vertebral-fracture-grader.git"
REPOSITORY_DIRECTORY = Path("/content/vertebral-fracture-grader")

if REPOSITORY_DIRECTORY.exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIRECTORY), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)],
        check=True,
    )

os.chdir(REPOSITORY_DIRECTORY)
print("Repository:", Path.cwd())


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements/colab.txt",
    ],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=True,
)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

TEMPORARY_WORK = Path("/content/medical-work")
RAW_DIRECTORY = TEMPORARY_WORK / "raw"
SEGMENTATION_DIRECTORY = TEMPORARY_WORK / "segmentations"

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/vertebral-fracture-grader"
)
CROP_DIRECTORY = DRIVE_OUTPUT / "crops"
MANIFEST_DIRECTORY = DRIVE_OUTPUT / "manifests"
QC_DIRECTORY = DRIVE_OUTPUT / "qc"

for directory in (
    RAW_DIRECTORY,
    SEGMENTATION_DIRECTORY,
    CROP_DIRECTORY,
    MANIFEST_DIRECTORY,
    QC_DIRECTORY,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Temporary raw data:", RAW_DIRECTORY)
print("Saved derived data:", DRIVE_OUTPUT)


In [ ]:
from localization.totalsegmentator_adapter import (
    build_totalsegmentator_command,
)

RUN_SEGMENTATION = False
CT_PATH = RAW_DIRECTORY / "replace-with-real-scan.nii.gz"
SCAN_SEGMENTATION_DIRECTORY = SEGMENTATION_DIRECTORY / "example-scan"

if RUN_SEGMENTATION:
    if not CT_PATH.is_file():
        raise FileNotFoundError(CT_PATH)

    command = build_totalsegmentator_command(
        CT_PATH,
        SCAN_SEGMENTATION_DIRECTORY,
        device="gpu",
    )
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
else:
    print(
        "Segmentation is disabled. Set CT_PATH and then change "
        "RUN_SEGMENTATION to True."
    )


In [ ]:
# Run this only after the Stage A segmentation cell succeeds.

from localization.totalsegmentator_adapter import (
    merge_binary_vertebra_masks,
)

MERGED_MASK_PATH = (
    SEGMENTATION_DIRECTORY / "example-scan-merged.nii.gz"
)

if RUN_SEGMENTATION:
    merge_binary_vertebra_masks(
        CT_PATH,
        SCAN_SEGMENTATION_DIRECTORY,
        MERGED_MASK_PATH,
    )
    print("Merged mask:", MERGED_MASK_PATH)


## Next action

1. Download one approved public dataset split into temporary storage.
2. Select one scan.
3. Set `CT_PATH`.
4. Change `RUN_SEGMENTATION` to `True`.
5. Inspect the segmentation and crop preview.
6. Do not begin full processing until the one-scan test passes.


In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(
    "/content/drive/MyDrive/vertebral-fracture-grader"
)

STEP_6B_FILES = [
    "model/dataset.py",
    "tests/test_model_dataset.py",
]

# Confirm the repository exists.
git_check = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "--is-inside-work-tree",
    ],
    capture_output=True,
    text=True,
)

if git_check.returncode != 0:
    raise RuntimeError(
        f"Not a valid Git repository: {REPO_DIR}\n"
        f"{git_check.stderr}"
    )

os.chdir(REPO_DIR)

# Confirm both Step 6B files exist.
missing_files = [
    file for file in STEP_6B_FILES
    if not Path(file).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing Step 6B files: {missing_files}"
    )

# Stage only the Step 6B files.
subprocess.run(
    ["git", "add", "--", *STEP_6B_FILES],
    check=True,
)

subprocess.run(
    ["git", "diff", "--cached", "--check"],
    check=True,
)

print("FILES STAGED:")
subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    check=True,
)

# Create the commit if changes are staged.
staged_status = subprocess.run(
    ["git", "diff", "--cached", "--quiet"]
).returncode

if staged_status == 1:
    subprocess.run(
        [
            "git",
            "-c",
            "user.name=Kushal Kilari",
            "-c",
            "user.email=kvskushal@users.noreply.github.com",
            "commit",
            "-m",
            "Add vertebra training dataset loader",
        ],
        check=True,
    )
elif staged_status == 0:
    print("\nNo new changes to commit.")
    print("Step 6B may already be committed.")
else:
    raise RuntimeError("Git could not inspect staged changes.")

print("\nLATEST LOCAL COMMIT:")
subprocess.run(
    ["git", "log", "-1", "--oneline"],
    check=True,
)

# Securely retrieve the token from Colab Secrets.
github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

# Send authentication without placing the token in the remote URL.
credentials = base64.b64encode(
    f"x-access-token:{github_token}".encode()
).decode()

auth_header = f"AUTHORIZATION: basic {credentials}"

print("\nPUSHING TO GITHUB...")

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader={auth_header}",
        "push",
        "origin",
        "HEAD:main",
    ],
    check=True,
)

print("\nVERIFYING GITHUB COMMIT...")

subprocess.run(
    ["git", "fetch", "origin", "main"],
    check=True,
)

local_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

github_commit = subprocess.run(
    ["git", "rev-parse", "origin/main"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

if local_commit != github_commit:
    raise RuntimeError(
        "Local commit and GitHub main do not match."
    )

print("Local commit: ", local_commit)
print("GitHub commit:", github_commit)
print("\nSTEP 6B COMMITTED AND PUSHED SUCCESSFULLY")